In [1]:
import yaml

param = {
    'operationmode': 'conventionalSequential',
    'deviceroadmap': 'HP',
    'transistortype': 'conventional',
    'readVoltage': 0.51,
    # /*** parameters for SRAM(parm.h) ***/
	# // due the scaling, suggested SRAM cell size above 22nm: 160F^2
	# // SRAM cell size at 14nm: 300F^2
	# // SRAM cell size at 10nm: 400F^2
	# // SRAM cell size at 7nm: 600F^2
    'heightInFeatureSizeSRAM': 10,  #SRAM Cell height in feature size
    'widthInFeatureSizeSRAM': 28,   #SRAM Cell width in feature size
    'widthSRAMCellNMOS': 2,
    'widthSRAMCellPMOS': 1,
    'widthAccessCMOS': 1,

    
    'minSenseVoltage': 0.1,
    
}

with open('../../param.yaml', 'w') as file:
    yaml.dump(param, file, default_flow_style=False)


In [2]:
with open('../../param.yaml', 'r') as file:
    loaded_config = yaml.safe_load(file)


print(loaded_config)


{'deviceroadmap': 'HP', 'heightInFeatureSizeSRAM': 10, 'minSenseVoltage': 0.1, 'operationmode': 'conventionalSequential', 'readVoltage': 0.51, 'transistortype': 'conventional', 'widthAccessCMOS': 1, 'widthInFeatureSizeSRAM': 28, 'widthSRAMCellNMOS': 2, 'widthSRAMCellPMOS': 1}


In [19]:
import math
import sys
import yaml
sys.path.append('../../python/')  
from periphery import logicGate
from periphery import constant
from periphery.Technology import Technology

def generate_param_yaml(
    filepath='param.yaml',
    validated = False,			#// false: no calibration factors
								#// true: validated by silicon data (wiring area in layout, gate switching activity, post-layout performance drop...)
    #/*** conventional hardware design options ***/
	temp = 300,                         #// Temperature (K)
	technode = 45,					    #// Technology node (nm)
    cell_type =  'SRAM',
    accesstype = 'CMOS_access'   # 'CMOS_access' or 'crossbar'

):
    
    numRowSubArray = 128;                # of rows in single subArray
    numColSubArray = 128;                # of columns in single subArray
    tech_table = {
        130: {'Metal0': 175, 'Metal1': 175, 'wireWidth': 175, 'barrierthickness': 10e-9, 'readVoltage': 0.58},
        90:  {'Metal0': 110, 'Metal1': 110, 'wireWidth': 110, 'barrierthickness': 10e-9, 'readVoltage': 0.55},
        65:  {'Metal0': 105, 'Metal1': 105, 'wireWidth': 105, 'barrierthickness': 7e-9,  'readVoltage': 0.51},
        45:  {'Metal0': 80,  'Metal1': 80,  'wireWidth': 80,  'barrierthickness': 5e-9,  'readVoltage': 0.51},
        32:  {'Metal0': 56,  'Metal1': 56,  'wireWidth': 56,  'barrierthickness': 4e-9,  'readVoltage': 0.51},
        22:  {'Metal0': 40,  'Metal1': 40,  'wireWidth': 40,  'barrierthickness': 2.5e-9,'readVoltage': 0.55},
        14:  {'Metal0': 32,  'Metal1': 39,  'wireWidth': 32,  'barrierthickness': 2.5e-9,'readVoltage': 0.277},
        10:  {'Metal0': 22,  'Metal1': 32,  'wireWidth': 22,  'barrierthickness': 2.0e-9,'readVoltage': 0.28},
        7:   {'Metal0': 20,  'Metal1': 28.5,'wireWidth': 20,  'barrierthickness': 2.0e-9,'readVoltage': 0.264},
        5:   {'Metal0': 15,  'Metal1': 17,  'wireWidth': 15,  'barrierthickness': 2.0e-9,'readVoltage': 0.253},
        3:   {'Metal0': 12,  'Metal1': 16,  'wireWidth': 12,  'barrierthickness': 1.5e-9,'readVoltage': 0.248},
        2:   {'Metal0': 10,  'Metal1': 11.5,'wireWidth': 10,  'barrierthickness': 0.5e-9, 'readVoltage': 0.28},
        1:   {'Metal0': 8,   'Metal1': 10,  'wireWidth': 8,   'barrierthickness': 0.2e-9, 'readVoltage': 0.272},
    }

    if technode not in tech_table:
        raise ValueError("Unsupported technology node")

    tech = tech_table[technode]
    wireWidth = tech['wireWidth']
    barrierthickness = tech['barrierthickness']
    readVoltage = tech['readVoltage']


    # /*** parameters for SRAM ***/
	# // due the scaling, suggested SRAM cell size above 22nm: 160F^2
	# // SRAM cell size at 14nm: 300F^2
	# // SRAM cell size at 10nm: 400F^2
	# // SRAM cell size at 7nm: 600F^2

    if technode > 14:
        heightInFeatureSizeSRAM = 10    #SRAM Cell height in feature size 
        widthInFeatureSizeSRAM = 28     #SRAM Cell width in feature size
        widthSRAMCellNMOS = 1
        widthSRAMCellPMOS = 1
        widthAccessCMOS = 1
    elif technode == 14:        # Samsung 14 nm
        heightInFeatureSizeSRAM = 10.6    #SRAM Cell height in feature size 
        widthInFeatureSizeSRAM = 30.8     #SRAM Cell width in feature size
        widthSRAMCellNMOS = 2
        widthSRAMCellPMOS = 1
        widthAccessCMOS = 1
    elif technode == 10:        # TSMC 10 nm
        heightInFeatureSizeSRAM = 12.8    #SRAM Cell height in feature size 
        widthInFeatureSizeSRAM = 31.25     #SRAM Cell width in feature size
        widthSRAMCellNMOS = 1
        widthSRAMCellPMOS = 1
        widthAccessCMOS = 1
    elif technode == 7:         # TSMC IEDM 2016
        heightInFeatureSizeSRAM = 16    #SRAM Cell height in feature size 
        widthInFeatureSizeSRAM = 34.43     #SRAM Cell width in feature size
        widthSRAMCellNMOS = 1
        widthSRAMCellPMOS = 1
        widthAccessCMOS = 1
    elif technode == 5:         # IRDS
        heightInFeatureSizeSRAM = 19.2    #SRAM Cell height in feature size 
        widthInFeatureSizeSRAM = 43.75     #SRAM Cell width in feature size
        widthSRAMCellNMOS = 1
        widthSRAMCellPMOS = 1
        widthAccessCMOS = 1
    elif technode == 3:         # IRDS
        heightInFeatureSizeSRAM = 30    #SRAM Cell height in feature size 
        widthInFeatureSizeSRAM = 68.26     #SRAM Cell width in feature size
        widthSRAMCellNMOS = 1
        widthSRAMCellPMOS = 1
        widthAccessCMOS = 1
    elif technode == 2:         # IRDS
        heightInFeatureSizeSRAM = 42    #SRAM Cell height in feature size 
        widthInFeatureSizeSRAM = 120     #SRAM Cell width in feature size
        widthSRAMCellNMOS = 1
        widthSRAMCellPMOS = 1
        widthAccessCMOS = 1
    elif technode == 1:         # IRDS
        heightInFeatureSizeSRAM = 80    #SRAM Cell height in feature size 
        widthInFeatureSizeSRAM = 144     #SRAM Cell width in feature size
        widthSRAMCellNMOS = 1
        widthSRAMCellPMOS = 1
        widthAccessCMOS = 1
    else:
        raise ValueError("Unsupported technology node")
    

    # /*** parameters for analog synaptic devices ***/
    heightInFeatureSize1T1R = 4        # 1T1R Cell height in feature size
    widthInFeatureSize1T1R = 12         # 1T1R Cell width in feature size
    heightInFeatureSizeCrossbar = 2    # Crossbar Cell height in feature size
    widthInFeatureSizeCrossbar = 2     # Crossbar Cell width in feature size
	
    resistanceOn = 6e3                   # 6e3;               // Ron resistance at Vr in the reported measurement data (need to recalculate below if considering the nonlinearity)
    resistanceOff = 6e3*17                 # 6e3*17;           // Roff resistance at Vr in the reported measurement dat (need to recalculate below if considering the nonlinearity)
    maxConductance =  1/resistanceOn
    minConductance =  1/resistanceOff  

    readPulseWidth = 10e-9;             # read pulse width in sec
    accessVoltage = 1.1;                # Gate voltage for the transistor in 1T1R
    resistanceAccess = resistanceOn*constant.IR_DROP_TOLERANCE;            # resistance of access CMOS in 1T1R
    writeVoltage = 2;					# Enable level shifer if writeVoltage > 1.5V

    levelOutput = 128;                   # # of levels of the multilevelSenseAmp output, should be in 2^N forms; e.g. 32 levels --> 5-bit ADC
    cellBit = 1;                        # precision of memory device 



    if (cell_type == "nvCap"): 		    # for cap array
        heightInFeatureSize1T1R = 2       # 1T1R Cell height in feature size
        widthInFeatureSize1T1R = 2        # 1T1R Cell width in feature size
        resistanceOn = 1e20               # Ron resistance at Vr in the reported measurement data (need to recalculate below if considering the nonlinearity)
        resistanceOff = 1e20*25        # Roff resistance = Ron * onoff ratio
        maxConductance =  1/resistanceOn
        minConductance =  1/resistanceOff
        readVoltage = 0.06	                # On-chip read voltage for memory cell
        accessVoltage = 0.06               # Gate voltage for the transistor in 1T1R
        chargeDelay = 5e-9				# Time to transfer charges to Cref



    # /*** Calibration parameters ***/
    # if(validated):
    alpha = 1.44	# wiring area of level shifter
    beta = 1.4  	# latency factor of sensing cycle
    gamma = 0.5 	# switching activity of DFF in shifter-add and accumulator
    delta = 0.15 	# switching activity of adder 
    epsilon = 0.05 # switching activity of control circuits
    zeta = 1.22 	# post-layout energy increase
	
	


    # /*** Initialize interconnect wires ***/
    # width AR Rho
    rho_table = [
        (175, 1.6, 2.01e-8), (110, 1.6, 2.20e-8), (105, 1.7, 2.21e-8), (80, 1.7, 2.37e-8),
        (56, 1.8, 2.63e-8), (40, 1.9, 2.97e-8), (32, 2.0, 3.25e-8), (22, 2.0, 3.95e-8),
        (20, 2.0, 4.17e-8), (15, 2.0, 4.98e-8), (12, 2.0, 5.8e-8), (10, 2.0, 6.61e-8), (8, 3.0, 7.87e-8)
    ]
    for width, AR, Rho in rho_table:
        if wireWidth >= width:
            break
    Rho_corrected = Rho / (1 - ((2 * AR * wireWidth + wireWidth) * barrierthickness / (AR * wireWidth ** 2)))
    Rho_corrected *= (1 + 0.00451 * abs(temp - 300))
    unit_length_wire_resistance = Rho_corrected / ((wireWidth * 1e-9) ** 2 * AR)

    Rho_Metal0 = Rho
    AR_Metal0 = AR
    Metal0 = tech['Metal0']
    Rho_Metal0 = Rho_Metal0 * 1 / (1- ( (2*AR_Metal0*Metal0 + Metal0)*barrierthickness / (AR_Metal0*pow(Metal0,2) ) ))

    Rho_Metal1 = Rho
    AR_Metal1 = AR
    Metal1 = tech['Metal1']
    Rho_Metal1 = Rho_Metal1 * 1 / (1- ( (2*AR_Metal1*Metal1 + Metal1)*barrierthickness / (AR_Metal1*pow(Metal1,2) ) ))

    Metal0_unitwireresis =  Rho_Metal0 / ( Metal0*1e-9 * Metal0*1e-9 * AR_Metal0 );
    Metal1_unitwireresis =  Rho_Metal1 / ( Metal1*1e-9 * Metal1*1e-9 * AR_Metal1 );
    

    if (cell_type == "SRAM"):
        wireLengthRowPerCell = wireWidth * 1e-9 * heightInFeatureSizeSRAM
        wireLengthColPerCell = wireWidth * 1e-9 * widthInFeatureSizeSRAM
        heightInFeatureSize = heightInFeatureSizeSRAM
        widthInFeatureSize = widthInFeatureSizeSRAM
    else:
        if (accesstype == "CMOS_access"):
            wireLengthRowPerCell = wireWidth * 1e-9 * heightInFeatureSize1T1R
            wireLengthColPerCell = wireWidth * 1e-9 * widthInFeatureSize1T1R
            heightInFeatureSize = heightInFeatureSize1T1R
            widthInFeatureSize = widthInFeatureSize1T1R
        else:
            wireLengthRowPerCell = wireWidth * 1e-9 * heightInFeatureSizeCrossbar
            wireLengthColPerCell = wireWidth * 1e-9 * widthInFeatureSizeCrossbar
            heightInFeatureSize = heightInFeatureSizeCrossbar
            widthInFeatureSize = widthInFeatureSizeCrossbar

    wireResistanceRowPerCell = unit_length_wire_resistance * wireLengthRowPerCell
    wireResistanceColPerCell = unit_length_wire_resistance * wireLengthColPerCell

    # Create param dictionary
    param = {
        'operationmode': 'conventionalSequential',
        'deviceroadmap': 'HP',
        'transistortype': 'conventional',
        'readVoltage': 0.51,
        'numRowSubArray': numRowSubArray,
        'numColSubArray': numColSubArray,
        # /*** parameters for SRAM(parm.h) ***/
	    # // due the scaling, suggested SRAM cell size above 22nm: 160F^2
	    # // SRAM cell size at 14nm: 300F^2
	    # // SRAM cell size at 10nm: 400F^2
	    # // SRAM cell size at 7nm: 600F^2
        'heightInFeatureSize': heightInFeatureSize,
        'widthInFeatureSize': widthInFeatureSize,
        'heightInFeatureSizeSRAM': heightInFeatureSizeSRAM,
        'widthInFeatureSizeSRAM': widthInFeatureSizeSRAM,
        'widthSRAMCellNMOS': 2,
        'widthSRAMCellPMOS': 1,
        'widthAccessCMOS': 1,

        # /*** parameters for analog synaptic devices ***/
        'resistanceOn': resistanceOn,                   # Ron resistance at Vr in the reported measurement data (need to recalculate below if considering the nonlinearity)
        'resistanceOff': resistanceOff,                 # Roff resistance at Vr in the reported measurement dat (need to recalculate below if considering the nonlinearity)
        'writeVoltage': writeVoltage,					# Enable level shifer if writeVoltage > 1.5V
        'accessVoltage': accessVoltage,                # Gate voltage for the transistor in 1T1R
        'cellBit': cellBit,                        # precision of memory device

        'minSenseVoltage': 0.1,

        'wireWidth': wireWidth,
        'barrierthickness': barrierthickness,
        'unitLengthWireResistance': unit_length_wire_resistance,
        'wireResistanceRowPerCell': wireResistanceRowPerCell,
        'wireResistanceColPerCell': wireResistanceColPerCell,

        'Metal0_unitwireresis': Metal0_unitwireresis,
        'Metal1_unitwireresis': Metal1_unitwireresis,

        'validated': False,			#// false: no calibration factors
        # /*** Calibration parameters ***/
        'alpha': alpha,
        'beta': beta,
        'gamma': gamma,
        'delta': delta,
        'epsilon': epsilon,
        'zeta': zeta

    }

    # Write to YAML
    with open(filepath, 'w') as file:
        yaml.dump(param, file, default_flow_style=False)

    print(f"param.yaml generated at: {filepath}")
    return param


In [21]:
generate_param_yaml(
    filepath='../../param.yaml',
    validated = False,			#// false: no calibration factors
								#// true: validated by silicon data (wiring area in layout, gate switching activity, post-layout performance drop...)
    #/*** conventional hardware design options ***/
	temp = 300,                         #// Temperature (K)
	technode = 45,					    #// Technology node (nm)
    cell_type =  'SRAM'

)

param.yaml generated at: ../../param.yaml


{'operationmode': 'conventionalSequential',
 'deviceroadmap': 'HP',
 'transistortype': 'conventional',
 'readVoltage': 0.51,
 'numRowSubArray': 128,
 'numColSubArray': 128,
 'heightInFeatureSize': 10,
 'widthInFeatureSize': 28,
 'heightInFeatureSizeSRAM': 10,
 'widthInFeatureSizeSRAM': 28,
 'widthSRAMCellNMOS': 2,
 'widthSRAMCellPMOS': 1,
 'widthAccessCMOS': 1,
 'resistanceOn': 6000.0,
 'resistanceOff': 102000.0,
 'writeVoltage': 2,
 'accessVoltage': 1.1,
 'cellBit': 1,
 'minSenseVoltage': 0.1,
 'wireWidth': 80,
 'barrierthickness': 5e-09,
 'unitLengthWireResistance': 2178308.8238817854,
 'wireResistanceRowPerCell': 1.7426470591054282,
 'wireResistanceColPerCell': 4.8794117654952,
 'Metal0_unitwireresis': 2178308.8238817854,
 'Metal1_unitwireresis': 2178308.8238817854,
 'validated': False,
 'alpha': 1.44,
 'beta': 1.4,
 'gamma': 0.5,
 'delta': 0.15,
 'epsilon': 0.05,
 'zeta': 1.22}